In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from notebooks.features.feature_extraction import load_all_features

features = load_all_features(version='oligo')
import pandas as pd

from notebooks.consts import OLIGO_CSV_INDEXED

data = pd.read_csv(OLIGO_CSV_INDEXED)

In [3]:
import pandas as pd

merged_df = pd.merge(features, data, on='index_oligo')

In [4]:
from notebooks.preprocessing import process_oligo_data

data = process_oligo_data(merged_df)

------------------------------------------------------------
PROCESSING FILTERING REPORT
------------------------------------------------------------
Initial raw rows loaded: 159,510

[0. BASE FILTERING]
Unsupported chemistry (Mixmers/DNA/None): 20,040
Steric blocking (True) eliminated: 0
Multiple genes (';' present) eliminated: 785
Missing inhibition (NaN) eliminated: 0

[1. UNMAPPED SEQUENCES (sense_start == -1)]
Samples eliminated: 821

[2. COHORT FILTERING (>= 50 samples)]
Cohorts: 318 -> 242 (76 eliminated)
Samples: 137,864 -> 135,481 (2,383 eliminated)

[3. SPARSE CELL LINE FILTERING (>= 1000 samples)]
Cell Lines: 27 -> 16 (11 eliminated)
Samples: 135,481 -> 128,220 (7,261 eliminated)

FINAL DATASET: 128,220 ASOs

ELIMINATED GROUPS BREAKDOWN

[ELIMINATED UNMAPPED SAMPLES] - 821 samples across 12 genes:
  • KCNQ2: 460 samples
  • IGF2-AS: 69 samples
  • ARL14EPP1: 67 samples
  • RP11-739B23.1: 59 samples
  • MIR6801: 54 samples
  • RP11-823E8.3: 52 samples
  • F12: 21 samples
  • 

In [5]:
from tauso.data.consts import *

# 3. Handle Transfection Features (Convert Boolean -> Int)
transfection_features = ['Electroporation', 'Gymnosis', 'Lipofection', 'Other']

# --- NEW LINE HERE ---
# Force conversion of True/False to 1/0
data[transfection_features] = data[transfection_features].astype(int)
# ---------------------

# 4. Define Final Feature List
features_to_ignore = ['index_oligo', INHIBITION, 'inhibition_percent', 'dosage']

# Note: Assuming 'merged_df' is your final dataframe. If you are just using 'data', change this to 'data'.
features = [
               col for col in data.select_dtypes(include=['number']).columns
               if col not in features_to_ignore
           ] + transfection_features + [VOLUME]

# Ensure uniqueness
features = sorted(list(set(features)))

In [6]:
# Create a DataFrame containing only your final chosen features
X = data[features]

# --- 1. Identify Constant Features (Zero Variance) ---
# nunique(dropna=False) <= 1 catches columns that are all the same value, or all NaNs
constant_features = [col for col in X.columns if X[col].nunique(dropna=False) <= 1]
print(f"⚠️ Found {len(constant_features)} constant features to drop: {constant_features}")

# Remove constant features from your final list
features = [f for f in features if f not in constant_features]
X = data[features]

# --- 2. Audit Features with NaNs ---
nan_counts = X.isna().sum()
features_with_nans = nan_counts[nan_counts > 0].sort_values(ascending=False)

print(f"\n⚠️ Found {len(features_with_nans)} features containing NaNs:")
for feature, count in features_with_nans.items():
    percent_missing = (count / len(X)) * 100
    print(f" - {feature}: {count} NaNs ({percent_missing:.1f}%)")

# --- 3. Control NaNs (Example Strategy) ---
# Let's say you only allow features with less than 10% missing data,
# and you want to explicitly review the ones that exceed that.
THRESHOLD = 10.0
broken_nan_features = []

for feature, count in features_with_nans.items():
    percent_missing = (count / len(X)) * 100
    if percent_missing > THRESHOLD:
        broken_nan_features.append(feature)

print(f"\n🚨 {len(broken_nan_features)} features exceeded the {THRESHOLD}% NaN threshold and should be dropped/imputed.")

# You can now cleanly remove the "broken" ones from your final list:
# features = [f for f in features if f not in broken_nan_features]

⚠️ Found 6 constant features to drop: ['METHYL_CYTOSINES', 'Modification_char_entropy', 'Modification_dominant_mod_fraction', 'OHE_pos20_C', 'OHE_pos20_G', 'OHE_pos20_T']

⚠️ Found 223 features containing NaNs:
 - CAI_score_20_CDS: 100070 NaNs (78.0%)
 - CAI_score_30_CDS: 100070 NaNs (78.0%)
 - CAI_score_40_CDS: 100070 NaNs (78.0%)
 - CAI_score_60_CDS: 100070 NaNs (78.0%)
 - CAI_score_50_CDS: 100070 NaNs (78.0%)
 - CAI_score_70_CDS: 100070 NaNs (78.0%)
 - ENC_score_30_CDS: 100052 NaNs (78.0%)
 - ENC_score_60_CDS: 100052 NaNs (78.0%)
 - ENC_score_20_CDS: 100052 NaNs (78.0%)
 - ENC_score_50_CDS: 100052 NaNs (78.0%)
 - ENC_score_40_CDS: 100052 NaNs (78.0%)
 - ENC_score_70_CDS: 100052 NaNs (78.0%)
 - tAI_score_60_CDS: 100052 NaNs (78.0%)
 - tAI_score_50_CDS: 100052 NaNs (78.0%)
 - tAI_score_40_CDS: 100052 NaNs (78.0%)
 - tAI_score_70_CDS: 100052 NaNs (78.0%)
 - tAI_score_20_CDS: 100052 NaNs (78.0%)
 - tAI_score_30_CDS: 100052 NaNs (78.0%)
 - cells_per_well: 9147 NaNs (7.1%)
 - RBP_diversit

In [7]:
from tauso.populate.populate_fold import populate_mfe_features
from tauso.genome.read_human_genome import get_locus_to_data_dict

na_data = data[data['access_120flank_20access_4-6-8seed_sizes'].isna()].copy()
na_data

,index_oligo,CAI_score_20_CDS,CAI_score_30_CDS,CAI_score_40_CDS,CAI_score_50_CDS,CAI_score_60_CDS,CAI_score_70_CDS,CAI_score_global_CDS,CET_DIFF_37_HYBR,DNA_HYBR_DIFF,...,rna_context,sugar_mods,backbone_mods,Canonical Gene Name,Chemical_Pattern,Modification,ps_pattern,cohort_id,Cell_Line_Depmap_Proxy,Cell_Line_Depmap
2565,2566,NaN,NaN,NaN,NaN,NaN,NaN,0.830285,-5.902,2059.753,...,AAAUUAGCAUGAAAGCAGUUUAGCAUUGGGAGGAAGCUCAGAUCUC...,"['CET', 'CET', 'CET', 'DNA', 'DNA', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",APOL1,CCCddddddddddCCC,cEt/5-methylcytosines/deoxy,***************,APOL1_A431,A-431,ACH-001328
3021,3022,NaN,NaN,NaN,NaN,NaN,NaN,0.832464,0.000,2515.939,...,ACCUCUUUCAAGUGCUGCCCUGGCUGAAGGAGAAACUCCAAGAUGA...,"['MOE', 'MOE', 'MOE', 'DNA', 'DNA', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",CFB,MMMddddddddddMMMMM,MOE/5-methylcytosines/deoxy,*****************,CFB_HepG2,Hep G2,ACH-000739
3022,3023,NaN,NaN,NaN,NaN,NaN,NaN,0.832464,0.000,2515.939,...,ACCUCUUUCAAGUGCUGCCCUGGCUGAAGGAGAAACUCCAAGAUGA...,"['MOE', 'MOE', 'MOE', 'MOE', 'MOE', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",CFB,MMMMMddddddddMMMMM,MOE/5-methylcytosines/deoxy,*****************,CFB_HepG2,Hep G2,ACH-000739
3023,3024,NaN,NaN,NaN,NaN,NaN,NaN,0.832464,0.000,2545.359,...,CCUCUUUCAAGUGCUGCCCUGGCUGAAGGAGAAACUCCAAGAUGAG...,"['MOE', 'MOE', 'MOE', 'DNA', 'DNA', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",CFB,MMMddddddddddMMMMM,MOE/5-methylcytosines/deoxy,*****************,CFB_HepG2,Hep G2,ACH-000739
3024,3025,NaN,NaN,NaN,NaN,NaN,NaN,0.832464,0.000,2545.359,...,CCUCUUUCAAGUGCUGCCCUGGCUGAAGGAGAAACUCCAAGAUGAG...,"['MOE', 'MOE', 'MOE', 'MOE', 'MOE', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",CFB,MMMMMddddddddMMMMM,MOE/5-methylcytosines/deoxy,*****************,CFB_HepG2,Hep G2,ACH-000739
14186,14187,NaN,NaN,NaN,NaN,NaN,NaN,0.801505,-7.033,2655.423,...,CAUUCCCUGAGCCCUGAUCCAUGCCUCAGCUUAGACUGCAGAGGAA...,"['CET', 'CET', 'CET', 'DNA', 'DNA', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",FOXP3,CCCddddddddddCCC,cEt/5-methylcytosines/deoxy,***************,FOXP3_LNCaP,LNCaP clone FGC,ACH-000977
15007,15008,NaN,NaN,NaN,NaN,NaN,NaN,0.832464,0.000,2307.398,...,ACCUCUUUCAAGUGCUGCCCUGGCUGAAGGAGAAACUCCAAGAUGA...,"['MOE', 'MOE', 'MOE', 'MOE', 'DNA', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",CFB,MMMMddddddddMMMMM,MOE/5-methylcytosines/deoxy,****************,CFB_HepG2,Hep G2,ACH-000739
15008,15009,NaN,NaN,NaN,NaN,NaN,NaN,0.832464,0.000,2456.812,...,CCUCUUUCAAGUGCUGCCCUGGCUGAAGGAGAAACUCCAAGAUGAG...,"['MOE', 'MOE', 'MOE', 'MOE', 'DNA', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",CFB,MMMMddddddddMMMMM,MOE/5-methylcytosines/deoxy,****************,CFB_HepG2,Hep G2,ACH-000739
15009,15010,NaN,NaN,NaN,NaN,NaN,NaN,0.832464,0.000,2446.353,...,CUCUUUCAAGUGCUGCCCUGGCUGAAGGAGAAACUCCAAGAUGAGG...,"['MOE', 'MOE', 'MOE', 'MOE', 'DNA', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",CFB,MMMMddddddddMMMMM,MOE/5-methylcytosines/deoxy,****************,CFB_HepG2,Hep G2,ACH-000739
15766,15897,NaN,NaN,NaN,NaN,NaN,NaN,0.830285,-6.794,2059.891,...,AAUUAGCAUGAAAGCAGUUUAGCAUUGGGAGGAAGCUCAGAUCUCU...,"['CET', 'CET', 'CET', 'DNA', 'DNA', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",APOL1,CCCddddddddddCCC,cEt/5-methylcytosines/deoxy,***************,APOL1_A431,A-431,ACH-001328


In [ ]:
gene_to_data = get_locus_to_data_dict(include_introns=True, gene_subset=na_data[CANONICAL_GENE].unique().tolist())

na_data2, features = populate_mfe_features(na_data, gene_to_data)
na_data2['mfe_win65_flank120_step15']

In [ ]:
from tauso.data.consts import SENSE_START, SENSE_LENGTH


def diagnose_nan_reason(row):
    gene_name = row[CANONICAL_GENE]
    global_start = row[SENSE_START]
    sense_len = row[SENSE_LENGTH]

    if gene_name not in gene_to_data:
        return "1. Gene not in gene_to_data (Dropped during loading?)"
    if global_start == -1:
        return "2. global_start is -1"

    locus_info = gene_to_data[gene_name]
    full_mrna = str(locus_info.full_mrna)

    # Simulating with one of the flanks/windows
    flank_size, window_size = 120, 65
    cut_start = max(0, global_start - flank_size)
    cut_end = min(len(full_mrna), global_start + sense_len + flank_size)
    sub_sequence = full_mrna[cut_start: cut_end]

    if len(sub_sequence) < window_size:
        return f"3. sub_sequence shorter than window_size ({len(sub_sequence)} < {window_size})"

    sequence_length = len(sub_sequence)
    sense_start_in_flank = global_start - cut_start
    sense_end_in_flank = sense_start_in_flank + sense_len

    if sense_start_in_flank < 0 or sense_start_in_flank >= sequence_length:
        return "4. sense_start out of sequence bounds"

    if sense_end_in_flank > sequence_length:
        return "5. sense_end out of bounds (feature extends past end of mRNA)"

    # Check if the sliding window missed the sense region completely
    # (Leaving counts == 0 -> np.nanmean on all NaNs -> returns NaN)
    counts = np.zeros(sequence_length)
    for i in range(0, sequence_length - window_size + 1, 15):
        counts[i:i + window_size] += 1

    sense_counts = counts[sense_start_in_flank:sense_end_in_flank]
    if np.all(sense_counts == 0):
        return "6. Sliding window step skipped the entire sense region"

    return "7. Unknown / ViennaRNA returned NaN"

# Apply and tally the reasons
na_data['nan_reason'] = na_data.apply(diagnose_nan_reason, axis=1)
print(na_data['nan_reason'].value_counts())

In [ ]:
import numpy as np
import networkx as nx

# 5. Fast Spearman Grouping Function
def get_correlated_groups(df, threshold=0.90):
    ranked_df = df.rank(na_option='keep')
    corr_matrix = ranked_df.corr(method='pearson').abs()
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    pairs = np.where(upper_tri > threshold)
    edges = [(corr_matrix.index[x], corr_matrix.columns[y]) for x, y in zip(*pairs)]

    G = nx.Graph()
    G.add_nodes_from(corr_matrix.columns)
    G.add_edges_from(edges)

    return [list(component) for component in nx.connected_components(G) if len(component) > 1]

# 6. Apply Correlation Filter
print(f"Original feature count: {len(features)}")
feature_df = data[features]

corr_groups = get_correlated_groups(feature_df, threshold=0.90)
features_to_drop = []

for group in corr_groups:
    # Calculate NaNs and Variance for the group
    nan_counts = feature_df[group].isna().sum()
    variances = feature_df[group].var()

    # Create a stat table to sort: Fewest NaNs first, Highest Variance second
    stats = pd.DataFrame({'nans': nan_counts, 'variance': variances})
    stats = stats.sort_values(by=['nans', 'variance'], ascending=[True, False])

    # The top index is our winner
    best_feature = stats.index[0]

    # Mark the rest for deletion
    drop_these = [f for f in group if f != best_feature]
    features_to_drop.extend(drop_these)

# 7. Finalize Dataset
features = [f for f in features if f not in features_to_drop]
data = data.drop(columns=features_to_drop)

print(f"Dropped {len(features_to_drop)} highly correlated features.")
print(f"Final feature count: {len(features)}")